In [1]:
# ============================================================
# DATEN LADEN – DWD Würzburg
# Zeitreihen: Wind (Kenia) | Luftdruck (Jonas) | Temperatur (Clara)
# Quelle: DWD CDC Open Data Server
# Cutoff: 01.01.2026
# ============================================================

import io
import re
import zipfile
import requests
import pandas as pd
from datetime import date

# ── Konfiguration ─────────────────────────────────────────────
STATION_ID = "05705"
BASE_URL   = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/"
CUTOFF     = date(2026, 1, 1)   # Festes Stoppdatum: 01.05.2026


# ── Hilfsfunktionen ───────────────────────────────────────────
def download_zip(url: str) -> zipfile.ZipFile:
    """Lädt eine ZIP-Datei von der DWD-URL und gibt sie als ZipFile zurück."""
    print(f"  → Lade: {url}")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    return zipfile.ZipFile(io.BytesIO(r.content))


def read_zip(zf: zipfile.ZipFile) -> pd.DataFrame:
    """Liest die Datendatei (produkt_*.txt) aus einem DWD-ZIP-Archiv."""
    data_file = [f for f in zf.namelist() if f.startswith("produkt_")][0]
    with zf.open(data_file) as f:
        return pd.read_csv(f, sep=";", encoding="latin-1")


def load_raw_data(station_id: str, base_url: str, cutoff: date) -> pd.DataFrame:
    """
    Lädt und kombiniert historische und aktuelle DWD-Daten für eine Station.
    Bereinigt Spaltennamen, setzt Datumsindex, ersetzt Fehlerwerte und
    schneidet am Cutoff-Datum ab.
    """
    print(f"\n{'='*55}")
    print(f"  Station: {station_id} | Cutoff: {cutoff}")
    print(f"{'='*55}")

    # Recent herunterladen
    print("\n[1/2] Aktuelle Daten:")
    zf_recent  = download_zip(f"{base_url}recent/tageswerte_KL_{station_id}_akt.zip")
    df_recent  = read_zip(zf_recent)

    # Historical herunterladen
    print("\n[2/2] Historische Daten:")
    base_hist  = base_url + "historical/"
    r          = requests.get(base_hist, timeout=30)
    pattern    = rf"tageswerte_KL_{station_id}_\d{{8}}_\d{{8}}_hist\.zip"
    filename   = re.findall(pattern, r.text)[0]
    zf_hist    = download_zip(base_hist + filename)
    df_hist    = read_zip(zf_hist)

    # Zusammenführen
    df = pd.concat([df_hist, df_recent], ignore_index=True)

    # Bereinigen
    df.columns     = df.columns.str.strip()
    df["MESS_DATUM"] = pd.to_datetime(df["MESS_DATUM"], format="%Y%m%d")
    df             = df.set_index("MESS_DATUM").sort_index()
    df             = df.replace(-999.0, float("nan"))
    df             = df.replace(-999,   float("nan"))
    df             = df[df.index.date <= cutoff]
    df             = df[~df.index.duplicated(keep="last")]

    return df


def summarize(name: str, df: pd.DataFrame):
    """Gibt eine kurze Zusammenfassung eines DataFrames aus."""
    print(f"\n  {name}:")
    print(f"    Zeitraum:       {df.index[0].date()} bis {df.index[-1].date()}")
    print(f"    Beobachtungen:  {len(df)}")
    print(f"    Fehlende Werte: {df.isna().sum().to_dict()}")


# ── Daten laden ───────────────────────────────────────────────
df_all_raw = load_raw_data(STATION_ID, BASE_URL, CUTOFF)


# ── Drei separate Zeitreihen extrahieren ──────────────────────
# Kenia – Wind
df_wind            = df_all_raw[["FM", "FX"]].copy()
df_wind.columns    = ["windgeschwindigkeit_ms", "windspitze_ms"]
df_wind            = df_wind.dropna(how="all")

# Jonas – Luftdruck
df_druck           = df_all_raw[["PM"]].copy()
df_druck.columns   = ["luftdruck_hpa"]
df_druck           = df_druck.dropna()

# Clara – Temperatur
df_temp            = df_all_raw[["TMK", "TXK", "TNK"]].copy()
df_temp.columns    = ["temperatur_mittel_c", "temperatur_max_c", "temperatur_min_c"]
df_temp            = df_temp.dropna(how="all")


# ── Zusammenfassung ───────────────────────────────────────────
print(f"\n{'='*55}")
print("  ÜBERSICHT DER ZEITREIHEN")
print(f"{'='*55}")
summarize("Wind          (Kenia)", df_wind)
summarize("Luftdruck     (Jonas)", df_druck)
summarize("Temperatur    (Clara)", df_temp)
print(f"\n{'='*55}")
print("  Daten erfolgreich geladen ✅")
print(f"{'='*55}\n")


  Station: 05705 | Cutoff: 2026-01-01

[1/2] Aktuelle Daten:
  → Lade: https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/recent/tageswerte_KL_05705_akt.zip

[2/2] Historische Daten:
  → Lade: https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/tageswerte_KL_05705_19470101_20241231_hist.zip

  ÜBERSICHT DER ZEITREIHEN

  Wind          (Kenia):
    Zeitraum:       1966-01-01 bis 2026-01-01
    Beobachtungen:  21799
    Fehlende Werte: {'windgeschwindigkeit_ms': 9, 'windspitze_ms': 1483}

  Luftdruck     (Jonas):
    Zeitraum:       1947-01-01 bis 2026-01-01
    Beobachtungen:  28848
    Fehlende Werte: {'luftdruck_hpa': 0}

  Temperatur    (Clara):
    Zeitraum:       1947-01-01 bis 2026-01-01
    Beobachtungen:  28856
    Fehlende Werte: {'temperatur_mittel_c': 0, 'temperatur_max_c': 90, 'temperatur_min_c': 90}

  Daten erfolgreich geladen ✅

